## task info

for titanic dataframe which exist in seaborn use decision tree and logistic regression and select the best one and tell me why you choose this model and use KNN and SVM must use gridsearchcv for KNN

 ## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

## Load dataset


In [2]:
df = sns.load_dataset("titanic")


## Select useful features


In [3]:
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]


## Drop rows with missing target


In [4]:
df = df.dropna(subset=["survived"])

X = df.drop("survived", axis=1)
y = df["survived"]


##  Numerical & categorical columns


In [5]:
num_cols = ["age", "fare", "sibsp", "parch"]
cat_cols = ["pclass", "sex", "embarked"]

## Preprocessing


In [6]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder",  pd.get_dummies)
])

## Using ColumnTransformer


In [7]:
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

## Train-test split


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Logistic Regression

In [9]:
log_model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)
log_acc = accuracy_score(y_test, log_pred)


## Decision Tree


In [10]:
tree_model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
tree_acc = accuracy_score(y_test, tree_pred)

## KNN (with GridSearchCV)

In [11]:
knn_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", KNeighborsClassifier())
])

param_grid = {
    "classifier__n_neighbors": [3, 5, 7, 9, 11],
    "classifier__weights": ["uniform", "distance"],
    "classifier__metric": ["euclidean", "manhattan"]
}

grid = GridSearchCV(
    knn_pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

knn_best = grid.best_estimator_
knn_pred = knn_best.predict(X_test)
knn_acc = accuracy_score(y_test, knn_pred)

## SVM

In [12]:
svm_model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", SVC())
])

svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_pred)


## Results

In [13]:
print("Logistic Regression Accuracy:", log_acc)
print("Decision Tree Accuracy:", tree_acc)
print("KNN (Best) Accuracy:", knn_acc)
print("SVM Accuracy:", svm_acc)

print("\nBest KNN Params:", grid.best_params_)

Logistic Regression Accuracy: 0.7988826815642458
Decision Tree Accuracy: 0.776536312849162
KNN (Best) Accuracy: 0.8324022346368715
SVM Accuracy: 0.8156424581005587

Best KNN Params: {'classifier__metric': 'manhattan', 'classifier__n_neighbors': 11, 'classifier__weights': 'uniform'}
